<a href="https://colab.research.google.com/github/Ebrahim827/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left">](https://colab.research.google.com/github/Ebrahim827/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# Week 4: Heuristic Baseline Action Score and Top-20 Review
This notebook implements a simple, hand-written business rule (a heuristic) to prioritize web pages for SEO content optimization.

## 1. My Rule and its Reason Codes
Our baseline targets pages that rank reasonably well on Google (Position <= 20) and have decent visibility (Impressions > 50) but suffer from a very low click-through rate (CTR < 2.0%).

We classify these priority pages into three actionable reason codes:
1. **`QUICK_WIN_TITLE`** (Priority: High | Score 90-100): Page ranks in the top 10 on Google but has a CTR below 1%. A minor metadata rewrite will yield immediate traffic spikes.
2. **`HIGH_IMP_LOW_CTR`** (Priority: Medium | Score 70-89): Page ranks in positions 11-20 with solid impressions but low CTR. Needs localized metadata optimization.
3. **`STALE_NO_TRAFFIC`** (Priority: Low | Score 30): Page is tracked but registered zero clicks over the entire month. Needs a deeper structural content audit.

In [24]:
!git clone https://github.com/Ebrahim827/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape[0], "rows x", df.shape[1], "cols")

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 209 (delta 81), reused 55 (delta 55), pack-reused 104 (from 1)
Receiving objects: 100% (209/209), 1.87 MiB | 6.91 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded: 30000 rows x 44 cols


In [25]:
bins = [-1, 30, 90, 180, 365, 10**6]
labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["_stale_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

sig_a = (
    df.groupby("_stale_bucket", observed=True)
      .agg(n=("content_id", "size"),
           pct_declining=("trend_direction", lambda s: (s == "down").mean()),
           mean_impressions_90d=("impressions_90d", "mean"))
      .reindex(labels)
)
print(f"Base rate % declining (all rows): {(df['trend_direction']=='down').mean():.3f}")
print(sig_a)


Base rate % declining (all rows): 0.542
                   n  pct_declining  mean_impressions_90d
_stale_bucket                                            
0-30           20480       0.511377           4199.614062
31-90            175       0.588571           6506.748571
91-180          9171       0.611057           7486.665140
181-365          169       0.467456           1206.893491
365+               5       0.600000              8.200000


Verdict: MIXED. The % of pages declining rises from 51.1% (0–30 days since update) to 61.1% (91–180 days, n=9,171 — a large, trustworthy sample), which supports the idea that older pages decline more. But it then drops to 46.7% for 181–365 days — below the overall average of 54.2% — and the 365+ day bucket only has 5 pages, far too few to trust. So staleness on its own is not a clean, reliable predictor of decline — it works in the early-to-mid range but breaks down for very old content. That's why the rule below doesn't rely on staleness alone; it also requires the page to still be getting real traffic.

In [26]:
bins_v = [-1, 100, 500, 2000, 10000, 10**7]
labels_v = ["0-100", "101-500", "501-2000", "2001-10000", "10000+"]
df["_volume_bucket"] = pd.cut(df["impressions_90d"], bins=bins_v, labels=labels_v)

sig_b = (
    df.groupby("_volume_bucket", observed=True)
      .agg(n=("content_id", "size"),
           pct_declining=("trend_direction", lambda s: (s == "down").mean()))
      .reindex(labels_v)
)
print(sig_b)

                   n  pct_declining
_volume_bucket                     
0-100           8006       0.389208
101-500         5279       0.604281
501-2000        6502       0.617964
2001-10000      6611       0.612918
10000+          3602       0.523598


Verdict: MIXED. Very low-volume pages (0–100 impressions, n=8,006) decline notably less often (38.9%) than the rest — likely because there's little traffic left to lose. But among pages with more than 100 impressions, the decline rate doesn't keep rising with volume — it plateaus around 60–62% (101 to 10,000 impressions) and then actually falls back to 52.4% for the highest-volume pages (10,000+, n=3,602), close to the overall base rate of 54.2%. So volume isn't a clean predictor that "more traffic = more at risk" — it only distinguishes very-low-traffic pages from everyone else.

In [27]:
STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 500

stale = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
visible = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)

df["score"] = stale * visible * df["impressions_90d"]
df["reason_code"] = np.where((stale == 1) & (visible == 1), "stale_but_visible", "no_flag")
df["action"] = np.where(df["score"] > 0, "refresh_review", "no_action")
print("Reason codes this rule can output:", df["reason_code"].unique())
print("Flagged rows:", (df["score"] > 0).sum(), "of", len(df))

Reason codes this rule can output: ['no_flag' 'stale_but_visible']
Flagged rows: 17 of 30000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [28]:
import os
os.makedirs("work/outputs", exist_ok=True)

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
out_cols = ["content_id", "client_id", "days_since_last_update", "impressions_90d",
            "ctr", "avg_position", "score", "reason_code", "action"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv —", len(queue), "rows")
queue[out_cols].head(20)

Wrote work/outputs/baseline_action_score.csv — 30000 rows


,content_id,client_id,days_since_last_update,impressions_90d,ctr,avg_position,score,reason_code,action
0,content_cf56e2e2e282,client_7f2253d7e2,194,61678,0.15,19.7,61678,stale_but_visible,refresh_review
1,content_7368877ea310,client_7f2253d7e2,194,59472,0.13,24.8,59472,stale_but_visible,refresh_review
2,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,0.23,22.2,25715,stale_but_visible,refresh_review
3,content_0a91db491d14,client_7f2253d7e2,193,13299,0.49,10.5,13299,stale_but_visible,refresh_review
4,content_5feee3994adb,client_7f2253d7e2,194,7812,0.01,39.0,7812,stale_but_visible,refresh_review
5,content_c2d929d83eaa,client_7f2253d7e2,193,7558,0.20,17.9,7558,stale_but_visible,refresh_review
6,content_b16bd7307b39,client_7f2253d7e2,194,4590,0.00,31.0,4590,stale_but_visible,refresh_review
7,content_fe16a55cd13d,client_7f2253d7e2,194,4556,0.33,16.4,4556,stale_but_visible,refresh_review
8,content_ecb6215e79fd,client_7f2253d7e2,194,4429,0.38,25.3,4429,stale_but_visible,refresh_review
9,content_928af3e22c80,client_7f2253d7e2,193,1697,0.12,15.8,1697,stale_but_visible,refresh_review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [29]:
top20 = queue.head(20)

def explain_row(row):
    bits = [f"{row['days_since_last_update']:.0f}d since update", f"{row['impressions_90d']:.0f} impr/90d"]
    if pd.notna(row.get("avg_position")):
        bits.append(f"avg position {row['avg_position']:.1f}")
    why = ", ".join(bits)

    caveats = []
    last30, prev30 = row.get("impressions_last_30d", np.nan), row.get("impressions_prev_30d", np.nan)
    if pd.notna(last30) and pd.notna(prev30) and prev30 > 0 and (last30/prev30) < 0.5:
        caveats.append("if the last-30d dip is a one-off rather than a sustained decline, this wastes a review")
    if pd.notna(row.get("word_count")) and row["word_count"] < 500:
        caveats.append("if this is a short utility page by design, staleness doesn't mean it needs more content")
    if row.get("main_intent") == "transactional":
        caveats.append("if the drop traces to price/competitor changes, a rewrite won't fix it")
    if row.get("competition_level") == "HIGH":
        caveats.append("if a competitor simply out-ranked this, refreshing may not win the position back")
    if not caveats:
        caveats.append("if traffic is seasonal and recovers on its own, flagging it now is premature")
    return why, caveats[0]

for i, row in top20.iterrows():
    why, wrong = explain_row(row)
    print(f"{i+1}. [{row['action']}] {row['content_id']} | reason: {row['reason_code']}")
    print(f"    why: {why}")
    print(f"    confidence: {'high' if row['impressions_90d'] > 5000 else 'medium'}")
    print(f"    would be wrong if: {wrong}")

1. [refresh_review] content_cf56e2e2e282 | reason: stale_but_visible
    why: 194d since update, 61678 impr/90d, avg position 19.7
    confidence: high
    would be wrong if: if the last-30d dip is a one-off rather than a sustained decline, this wastes a review
2. [refresh_review] content_7368877ea310 | reason: stale_but_visible
    why: 194d since update, 59472 impr/90d, avg position 24.8
    confidence: high
    would be wrong if: if the last-30d dip is a one-off rather than a sustained decline, this wastes a review
3. [refresh_review] content_1bfaa38ff26c | reason: stale_but_visible
    why: 194d since update, 25715 impr/90d, avg position 22.2
    confidence: high
    would be wrong if: if the last-30d dip is a one-off rather than a sustained decline, this wastes a review
4. [refresh_review] content_0a91db491d14 | reason: stale_but_visible
    why: 193d since update, 13299 impr/90d, avg position 10.5
    confidence: high
    would be wrong if: if the last-30d dip is a one-off rather

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [30]:
flagged = queue[queue["score"] > 0].copy()
if len(flagged) == 0:
    print("No rows cleared the bar — check STALE_DAYS/VISIBLE_IMPRESSIONS against Signal A's table.")
else:
    weakest = flagged.nsmallest(3, "impressions_90d")
    for i, row in weakest.iterrows():
        print(f"- {row['content_id']}: only {row['impressions_90d']:.0f} impr/90d — barely clears the "
              f"visibility bar, so this pick is fragile, not a clear win.")

print("\nLeakage check:")
rule_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
print("Rule inputs used:", rule_inputs)
print("Forbidden label/future columns NOT used as rule inputs:", forbidden & set(df.columns))

- content_074ba6ead17b: only 533 impr/90d — barely clears the visibility bar, so this pick is fragile, not a clear win.
- content_6226ee6adc91: only 545 impr/90d — barely clears the visibility bar, so this pick is fragile, not a clear win.
- content_72496874f806: only 821 impr/90d — barely clears the visibility bar, so this pick is fragile, not a clear win.

Leakage check:
Rule inputs used: {'days_since_last_update', 'impressions_90d'}
Forbidden label/future columns NOT used as rule inputs: {'trend_pct', 'trend_direction'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.